# 01 - Authenticate

Resolve an authentication **strategy** from a flat credential bundle and obtain an
`Authorization` header. This example uses the OAuth **client-credentials** grant
against Account Manager -- the token endpoint is mocked with `respx`, so it runs
fully offline with fake credentials.

Public API: `resolve_auth_strategy`, `AuthCredentials`, `OAuthStrategy`.

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

In [ ]:
# The SDK decodes (does NOT verify) Account Manager access tokens, so a valid
# JWT *shape* is enough for a mocked token endpoint. This mirrors the unsigned
# test JWTs used by the SDK's own suite (tests/helpers/jwt.py).
def make_jwt(*, expires_in=3600, scope=None, sub=None):
    def b64(raw: bytes) -> str:
        return base64.urlsafe_b64encode(raw).rstrip(b"=").decode("ascii")

    header = {"alg": "RS256", "typ": "JWT"}
    payload = {"exp": int(time.time()) + expires_in}
    if scope is not None:
        payload["scope"] = scope
    if sub is not None:
        payload["sub"] = sub
    return ".".join([b64(json.dumps(header).encode()), b64(json.dumps(payload).encode()), "sig"])

## Resolve a strategy

`resolve_auth_strategy` inspects the credentials and returns the first strategy
whose requirements are met (here: client id + secret -> client-credentials).

In [ ]:
from b2c_tooling_sdk import (
    DEFAULT_ACCOUNT_MANAGER_HOST,
    AuthCredentials,
    OAuthStrategy,
    resolve_auth_strategy,
)

credentials = AuthCredentials(
    client_id="my-client-id",
    client_secret="my-client-secret",
    scopes=["sfcc.products"],
)

strategy = resolve_auth_strategy(credentials)
print("Resolved strategy:", type(strategy).__name__)
assert isinstance(strategy, OAuthStrategy)

TOKEN_URL = f"https://{DEFAULT_ACCOUNT_MANAGER_HOST}/dwsso/oauth2/access_token"
print("Account Manager token endpoint:", TOKEN_URL)

## Obtain an authorization header

`get_authorization_header()` performs the client-credentials grant (mocked here)
and returns a ready-to-use `Bearer` header. The token is cached on the strategy.

In [ ]:
with respx.mock(assert_all_called=False) as router:
    router.post(TOKEN_URL).mock(
        return_value=httpx.Response(
            200,
            json={
                "access_token": make_jwt(scope="sfcc.products", sub="my-client-id"),
                "expires_in": 1800,
                "scope": "sfcc.products",
            },
        )
    )

    header = await strategy.get_authorization_header()

assert header.startswith("Bearer ")
print("Authorization:", header[:24] + "...")

## Recap

- `resolve_auth_strategy(AuthCredentials(...))` picked the client-credentials
  `OAuthStrategy` automatically.
- `await strategy.get_authorization_header()` minted (and cached) a bearer token.
- No real Account Manager call was made -- the token endpoint was mocked.